## 1. Load libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## 2. Load data

In [2]:
analysis_df = pd.read_csv(
    "../data/processed/final/music_taste_analysis_ready.csv"
)

## 3. Define analysis period 2008-2025

In [20]:
# Exclude incomplete years from the analysis

analysis_df = analysis_df[
    (analysis_df["year_file"] >= 2008)
    & (analysis_df["year_file"] <= 2025)
].copy()

In [21]:
# Top artist by Era

def assign_era(year):
    if year <= 2011:
        return "2008-2011"
    elif year <= 2016:
        return "2012-2016"
    elif year <= 2021:
        return "2017-2021"
    else:
        return "2022-2025"

analysis_df["era"] = analysis_df["year_file"].apply(assign_era)

## 4. Inspect dataset structure

In [22]:
# Distribution of the analytical dataset across macro-clusters and eras

macro_era_counts = (
    analysis_df
    .groupby(["macro_cluster", "era"])
    .size()
    .unstack(fill_value=0)
)

macro_era_counts

era,2008-2011,2012-2016,2017-2021,2022-2025
macro_cluster,,,,
Alternative Core,2497,7147,14750,11348
Electronic & Ambient,835,3092,6530,5425
Rock & Britrock,1953,3384,6144,3247
Singer-Songwriter,545,1039,1156,1125


In [23]:
#Jjaki procent całego datasetu stanowi dana kombinacja era × macro-cluster.

macro_era_pct = (
    macro_era_counts
    .div(macro_era_counts.sum().sum())
    * 100
)

macro_era_pct.round(1)

era,2008-2011,2012-2016,2017-2021,2022-2025
macro_cluster,,,,
Alternative Core,3.6,10.2,21.0,16.2
Electronic & Ambient,1.2,4.4,9.3,7.7
Rock & Britrock,2.8,4.8,8.8,4.6
Singer-Songwriter,0.8,1.5,1.6,1.6


In [14]:
# There are no empty layers in the macro-cluster × era distribution, so we can proceed with the analysis.

## 4. Build a stratified sample
To assess lyrics coverage, a stratified sample is drawn from the analytical dataset.

The sample is stratified by era and macro-cluster to ensure that all combinations of these two dimensions are represented equally.

A fixed number of 50 unique artist–track combinations is sampled from each stratum, resulting in a total sample of 800 records.

In [24]:
n_per_stratum = 50

lyrics_sample = (
    analysis_df
    .groupby(["era", "macro_cluster"], group_keys=False)
    .sample(n=n_per_stratum, random_state=42) #random_state=42 ensures reproducibility
    .reset_index(drop=True)
)

lyrics_sample.shape

(800, 16)

In [25]:
lyrics_sample.groupby(["era", "macro_cluster"]).size().unstack(fill_value=0)

macro_cluster,Alternative Core,Electronic & Ambient,Rock & Britrock,Singer-Songwriter
era,,,,
2008-2011,50,50,50,50
2012-2016,50,50,50,50
2017-2021,50,50,50,50
2022-2025,50,50,50,50


## 5. Prepare the lyrics coverage sample

The sampled dataset contains 800 unique artist–track combinations selected through stratified sampling across eras and macro-clusters.

The sample will be used to evaluate lyrics availability and identify potential coverage bias across different periods and musical groups.

In [27]:
lyrics_sample[
    [
        "artist",
        "track",
        "artist_clean",
        "track_clean",
        "year_file",
        "era",
        "macro_cluster"
    ]
].head(20)

,artist,track,artist_clean,track_clean,year_file,era,macro_cluster
0,Squarepusher,04 Tommib,squarepusher,04 tommib,2011,2008-2011,Alternative Core
1,Angels & Airwaves,Secret Crowds,angels airwaves,secret crowds,2011,2008-2011,Alternative Core
2,! www.polskie-mp3.tk ! ewa demarczyk,02. garbus,wwwpolskiemp3tk ewa demarczyk,02 garbus,2008,2008-2011,Alternative Core
3,Red Hot Chili Peppers,Monarchy of Roses,red hot chili peppers,monarchy of roses,2011,2008-2011,Alternative Core
4,Kasia Nosowska,Poli D.N.O.,kasia nosowska,poli dno,2009,2008-2011,Alternative Core
5,Tomek Makowiecki,Nie boje sie,tomek makowiecki,nie boje sie,2008,2008-2011,Alternative Core
6,Brandon Flowers,Only the Young,brandon flowers,only the young,2011,2008-2011,Alternative Core
7,T.Love,Gwiazdka,tlove,gwiazdka,2008,2008-2011,Alternative Core
8,Pennywise,Living for Today,pennywise,living for today,2008,2008-2011,Alternative Core
9,! www.polskie-mp3.tk ! ewa demarczyk,01. karuzela madonnami,wwwpolskiemp3tk ewa demarczyk,01 karuzela madonnami,2008,2008-2011,Alternative Core


## 6. LRCLIB coverage pilot

A small pilot sample is used to evaluate LRCLIB lyrics availability before querying the full sample.

Three tracks are selected from each era × macro-cluster stratum, resulting in 48 tracks in total.

In [28]:
# Draw a small stratified pilot sample
pilot_sample = (
    lyrics_sample
    .groupby(["era", "macro_cluster"], group_keys=False)
    .sample(n=3, random_state=42)
    .reset_index(drop=True)
)

pilot_sample.shape

(48, 16)

In [29]:
pilot_sample.groupby(
    ["era", "macro_cluster"]
).size().unstack(fill_value=0)

macro_cluster,Alternative Core,Electronic & Ambient,Rock & Britrock,Singer-Songwriter
era,,,,
2008-2011,3,3,3,3
2012-2016,3,3,3,3
2017-2021,3,3,3,3
2022-2025,3,3,3,3


## 7. Test LRCLIB lyrics coverage

LRCLIB is queried using the cleaned artist and track names. The returned artist and track names are retained to validate whether the result corresponds to the requested recording.

In [32]:
import requests
import time
import re

In [33]:
# separate function to normalize text for matching, we don't change anything in the original columns
def normalize_for_match(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "", text)
    return text

In [49]:
def search_lrclib(artist, track, max_retries=3):
    url = "https://lrclib.net/api/search"

    params = {
        "artist_name": artist,
        "track_name": track
    }

    headers = {
        "User-Agent": "music-mood-profiler/1.0"
    }

    for attempt in range(max_retries):
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=10
        )

        if response.status_code == 404:
            return []

        if response.status_code == 503:
            if attempt < max_retries - 1:
                time.sleep(2)
                continue

            return None

        response.raise_for_status()

        return response.json()

    return None

In [50]:
def check_lrclib_match(artist, track):
    results = search_lrclib(artist, track)

    if results is None:
        return {
            "status": "temporary_error",
            "matched_artist": None,
            "matched_track": None,
            "instrumental": None,
            "has_plain_lyrics": None,
            "has_synced_lyrics": None
        }

    if not results:
        return {
            "status": "not_found",
            "matched_artist": None,
            "matched_track": None,
            "instrumental": None,
            "has_plain_lyrics": None,
            "has_synced_lyrics": None
        }

    artist_norm = normalize_for_match(artist)
    track_norm = normalize_for_match(track)

    for result in results:
        result_artist = normalize_for_match(
            result.get("artistName", "")
        )

        result_track = normalize_for_match(
            result.get("trackName", "")
        )

        if (
            result_artist == artist_norm
            and result_track == track_norm
        ):
            return {
                "status": "found",
                "matched_artist": result.get("artistName"),
                "matched_track": result.get("trackName"),
                "instrumental": result.get("instrumental"),
                "has_plain_lyrics": bool(result.get("plainLyrics")),
                "has_synced_lyrics": bool(result.get("syncedLyrics"))
            }

    return {
        "status": "ambiguous",
        "matched_artist": None,
        "matched_track": None,
        "instrumental": None,
        "has_plain_lyrics": None,
        "has_synced_lyrics": None
    }

In [51]:
for i in [0, 1, 2, 3, 5, 9]:
    artist = pilot_sample.iloc[i]["artist_clean"]
    track = pilot_sample.iloc[i]["track_clean"]

    result = check_lrclib_match(artist, track)

    print(
        f"{i}: {artist} — {track} → "
        f"{result['status']}"
    )

0: makowiecki band — bullet in my back → not_found
1: a silent film — driven by their beating hearts → found
2: eric clapton — layla → found
3: foo fighters — walk → found
5: cat power — fiance → not_found
9: massive attack — be thankful for what you got → found


In [52]:
pilot_results = []

for _, row in pilot_sample.iterrows():
    result = check_lrclib_match(
        row["artist_clean"],
        row["track_clean"]
    )

    pilot_results.append({
        "artist": row["artist"],
        "track": row["track"],
        "artist_clean": row["artist_clean"],
        "track_clean": row["track_clean"],
        "year_file": row["year_file"],
        "era": row["era"],
        "macro_cluster": row["macro_cluster"],
        **result
    })

    time.sleep(0.5)

pilot_results_df = pd.DataFrame(pilot_results)

In [53]:
pilot_results_df["status"].value_counts()

status
found        30
not_found    16
ambiguous     2
Name: count, dtype: int64

In [54]:
pilot_results_df.groupby(
    ["era", "status"]
).size().unstack(fill_value=0)

status,ambiguous,found,not_found
era,,,
2008-2011,0,9,3
2012-2016,1,6,5
2017-2021,1,7,4
2022-2025,0,8,4


In [55]:
pilot_results_df.groupby(
    ["macro_cluster", "status"]
).size().unstack(fill_value=0)

status,ambiguous,found,not_found
macro_cluster,,,
Alternative Core,2,5,5
Electronic & Ambient,0,11,1
Rock & Britrock,0,8,4
Singer-Songwriter,0,6,6


In [56]:
pilot_results_df[
    pilot_results_df["status"] == "ambiguous"
][[
    "artist",
    "track",
    "artist_clean",
    "track_clean",
    "era",
    "macro_cluster"
]]

,artist,track,artist_clean,track_clean,era,macro_cluster
12,OCEAN - OCN,The First Cut,ocean,the first cut,2012-2016,Alternative Core
26,Ludwig van Beethoven,"Piano Concerto No. 3 in C minor, Op. 37: I. Al...",ludwig van beethoven,piano concerto no 3 in c minor op 37 i allegro...,2017-2021,Alternative Core


In [57]:
for i in [12, 26]:
    artist = pilot_sample.iloc[i]["artist_clean"]
    track = pilot_sample.iloc[i]["track_clean"]

    results = search_lrclib(artist, track)

    print(f"\n--- {i}: {artist} — {track} ---")

    if results is None:
        print("503 - server unavailable")
        continue

    if not results:
        print("No results")
        continue

    for result in results:
        print(
            f"{result.get('artistName')} — "
            f"{result.get('trackName')} — "
            f"album: {result.get('albumName')} — "
            f"instrumental: {result.get('instrumental')}"
        )


--- 12: ocean — the first cut ---
Nikki Ocean — The First Cut Is the Deepest — album: Midnight in Paris — instrumental: False

--- 26: ludwig van beethoven — piano concerto no 3 in c minor op 37 i allegro con brio ---
Ludwig van Beethoven; Daniel Barenboim — Piano Concerto no. 3 in C minor, op. 37: I. Allegro con brio — album: Beethoven for All: The Piano Concertos — instrumental: True
Ludwig van Beethoven; Krystian Zimerman, London Symphony Orchestra, Simon Rattle — Piano Concerto no. 3 in C minor, op. 37: I. Allegro con brio — album: Complete Piano Concertos — instrumental: True
Ludwig van Beethoven; Arthur Rubinstein, London Philharmonic Orchestra, Daniel Barenboim — Piano Concerto No. 3 in C minor, Op. 37: I. Allegro con brio — album: The Rubinstein Collection, Volume 78: Piano Concertos Nos. 3 & 4 — instrumental: True
Ludwig van Beethoven/Krystian Zimerman/London Symphony Orchestra/Sir Simon Rattle — Piano Concerto No. 3 in C Minor, Op. 37: I. Allegro con brio — album: Beethoven:

In [41]:
test_artist = pilot_sample.iloc[0]["artist_clean"]
test_track = pilot_sample.iloc[0]["track_clean"]

results = search_lrclib(
    test_artist,
    test_track
)

print(test_artist)
print(test_track)
print(results)

makowiecki band
bullet in my back
[]


In [42]:
results = search_lrclib(
    test_artist,
    test_track
)

print(results)

[]


In [43]:
test_artist = pilot_sample.iloc[1]["artist_clean"]
test_track = pilot_sample.iloc[1]["track_clean"]

results = search_lrclib(
    test_artist,
    test_track
)

print(test_artist)
print(test_track)
print(results)

a silent film
driven by their beating hearts
[{'id': 18810924, 'name': 'Driven by Their Beating Hearts', 'trackName': 'Driven by Their Beating Hearts', 'artistName': 'A Silent Film', 'albumName': 'The City That Sleeps', 'duration': 258.368435, 'instrumental': False, 'plainLyrics': "As I was waking up\nThe moon was fading\nThe fox had gone to ground\nThe day was breaking\n\nWords don't come easily\nMost when I need them\nI do not have a key\nI am breaking in\n\nThere's people going out of their minds\nRunning to each other's arms\nBut I only want to be with you\nThere's people getting out of their cars\nDriven by their beating hearts\nBut I only want to be with you\n\nI don't believe my eyes\nThere's so much beauty here\nA song for everyone\nA sound for every ear\n\nI want you to define me", 'syncedLyrics': "[00:30.82] As I was waking up\n[00:32.49] The moon was fading\n[00:34.37] The fox had gone to ground\n[00:36.44] The day was breaking\n[00:38.06] Words don't come easily\n[00:39.66]

In [44]:
result = results[0]

print("ID:", result.get("id"))
print("Artist:", result.get("artistName"))
print("Track:", result.get("trackName"))
print("Album:", result.get("albumName"))
print("Duration:", result.get("duration"))
print("Instrumental:", result.get("instrumental"))
print("Has plain lyrics:", bool(result.get("plainLyrics")))
print("Has synced lyrics:", bool(result.get("syncedLyrics")))

ID: 18810924
Artist: A Silent Film
Track: Driven by Their Beating Hearts
Album: The City That Sleeps
Duration: 258.368435
Instrumental: False
Has plain lyrics: True
Has synced lyrics: True


In [45]:
pilot_sample[["artist_clean", "track_clean"]].head(10)

,artist_clean,track_clean
0,makowiecki band,bullet in my back
1,a silent film,driven by their beating hearts
2,eric clapton,layla
3,foo fighters,walk
4,jack johnson,holes to heaven
5,cat power,fiance
6,tegan and sara,freedom
7,kids of 88,nerves
8,iamx,this will make you love again
9,massive attack,be thankful for what you got


In [46]:
for i in [2, 3, 5, 9]:
    artist = pilot_sample.iloc[i]["artist_clean"]
    track = pilot_sample.iloc[i]["track_clean"]

    results = search_lrclib(artist, track)

    print(f"{i}: {artist} — {track}")
    print(f"Number of results: {len(results) if results is not None else '503'}")
    print()

2: eric clapton — layla
Number of results: 20

3: foo fighters — walk
Number of results: 20

5: cat power — fiance
Number of results: 0

9: massive attack — be thankful for what you got
Number of results: 8



In [47]:
for i in [2, 3, 5, 9]:
    artist = pilot_sample.iloc[i]["artist_clean"]
    track = pilot_sample.iloc[i]["track_clean"]

    results = search_lrclib(artist, track)

    print(f"\n--- {i}: {artist} — {track} ---")

    if results is None:
        print("503 - server unavailable")
        continue

    if not results:
        print("No results")
        continue

    for result in results[:10]:
        print(
            f"{result.get('artistName')} — "
            f"{result.get('trackName')} — "
            f"album: {result.get('albumName')}"
        )


--- 2: eric clapton — layla ---
Eric Clapton — Layla — album: Unplugged
Eric Clapton — Layla — album: [2021] The Lady In The Balcony: Lockdown Sessions
Eric Clapton — Layla — album: Complete Clapton
Eric Clapton — Layla — album: link(???)
Eric Clapton — Layla — album: At His Best (Reel to Reel 7.5 ips)
Eric Clapton — Layla — album: Bottom Dollar
ERIC CLAPTON — LAYLA — album: Exclusive Rock & Rock Ballads 2
Eric Clapton — Layla — album: One More Car, One More Rider [Enhanced Version] (2 of 2)
Eric Clapton — Layla — album: Mp3
Eric Clapton — Layla — album: Layla & Assorted Love Songs

--- 3: foo fighters — walk ---
Foo Fighters — Walk;Walk — album: Arrow Rock 500 Editie 2015
Foo Fighters — Walk — album: Top Hits 2012
Foo Fighters — Walk — album: 2014-09-14: Queen Elizabeth Olympic Park, London, England
Foo Fighters — Walk — album: 2015-05-24: BBC Radio 1's Big Weekend Festival, England
Foo Fighters — Walk — album: 2017-10-28: Voodoo Music + Arts Experience Festival, New Orleans, LA, USA